In [1]:
import pandas as pd
import numpy as np

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.model_selection import (
    StratifiedKFold,
    cross_validate,
    train_test_split,
)
from sklearn.metrics import confusion_matrix, f1_score, make_scorer

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB, BernoulliNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    ConfusionMatrixDisplay, roc_auc_score, average_precision_score
)

## Read Data CSV

In [2]:
sessions = pd.read_csv("Raw_Data/challenge2_train.csv")

display(sessions.head())
sessions.info()

print()
display(sessions["Revenue"].value_counts(dropna=False))
print()
display(sessions["Revenue"].value_counts(normalize=True, dropna=False))

,ID,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,1,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,1,1,1,1,Returning_Visitor,False,False
1,3,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,4,1,9,3,Returning_Visitor,False,False
2,4,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,Feb,3,2,2,4,Returning_Visitor,False,False
3,7,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.4,Feb,2,4,3,3,Returning_Visitor,False,False
4,9,0,0.0,0,0.0,2,37.000000,0.00,0.10,0.0,0.8,Feb,2,2,2,3,Returning_Visitor,False,False


<class 'pandas.DataFrame'>
RangeIndex: 9247 entries, 0 to 9246
Data columns (total 19 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   ID                       9247 non-null   int64  
 1   Administrative           9247 non-null   int64  
 2   Administrative_Duration  9247 non-null   float64
 3   Informational            9247 non-null   int64  
 4   Informational_Duration   9247 non-null   float64
 5   ProductRelated           9247 non-null   int64  
 6   ProductRelated_Duration  9247 non-null   float64
 7   BounceRates              9247 non-null   float64
 8   ExitRates                9247 non-null   float64
 9   PageValues               9247 non-null   float64
 10  SpecialDay               9247 non-null   float64
 11  Month                    9247 non-null   str    
 12  OperatingSystems         9247 non-null   int64  
 13  Browser                  9247 non-null   int64  
 14  Region                   9247 non-n

Revenue
False    7816
True     1431
Name: count, dtype: int64

Revenue
False    0.845247
True     0.154753
Name: proportion, dtype: float64

## Train/Validation Splits 

In [3]:
X = sessions.drop(columns=["Revenue", "ID"])
y = sessions["Revenue"].copy()

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

display(pd.DataFrame({
    "development": y_train.value_counts(normalize=True),
    "holdout": y_valid.value_counts(normalize=True),
}))

,development,holdout
Revenue,,
False,0.845208,0.845405
True,0.154792,0.154595


## Model Pipeline

In [4]:
categorical = [
    "Month",
    "OperatingSystems",
    "Browser",
    "Region",
    "TrafficType",
    "VisitorType",
    "Weekend",
]

numerical = [column for column in X_train.columns if column not in categorical]

# Pipeline Preprocess 
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical)
    ]
)

### Logistic Regression

In [5]:
logistic_pipeline = Pipeline([
    ("prep", preprocessor),
    ("model", LogisticRegression(max_iter=1000)),
])

fitted_logistic = clone(logistic_pipeline).fit(X_train, y_train)

feature_names = fitted_logistic.named_steps["prep"].get_feature_names_out()
weights = fitted_logistic.named_steps["model"].coef_[0]

logistic_weights = pd.DataFrame({
    "Feature": feature_names,
    "Weight": weights,
})

display(logistic_weights.round(3))

,Feature,Weight
0,num__Administrative,-0.050
1,num__Administrative_Duration,0.030
2,num__Informational,0.022
3,num__Informational_Duration,0.022
4,num__ProductRelated,0.170
...,...,...
68,cat__VisitorType_New_Visitor,-0.043
69,cat__VisitorType_Other,-0.389
70,cat__VisitorType_Returning_Visitor,-0.181
71,cat__Weekend_False,-0.375


### Naïve Bayes

In [6]:
Bayes_pipe = Pipeline([
    ("prep", preprocessor),
    ("clf", BernoulliNB())
])

Bayes_pipe.fit(X_train, y_train)

feature_names = Bayes_pipe.named_steps['prep'].get_feature_names_out()

feature_prob = np.exp(Bayes_pipe.named_steps['clf'].feature_log_prob_)

learned_probabilities = pd.DataFrame({
    'Feature': feature_names,
    'P(Feature=1 | y=0)': feature_prob[0],
    'P(Feature=1 | y=1)': feature_prob[1]
})

learned_probabilities.head(10).round(2)


,Feature,P(Feature=1 | y=0),P(Feature=1 | y=1)
0,num__Administrative,0.31,0.48
1,num__Administrative_Duration,0.25,0.40
2,num__Informational,0.19,0.33
3,num__Informational_Duration,0.12,0.22
4,num__ProductRelated,0.27,0.47
5,num__ProductRelated_Duration,0.28,0.48
6,num__BounceRates,0.23,0.04
7,num__ExitRates,0.33,0.05
8,num__PageValues,0.08,0.72
9,num__SpecialDay,0.12,0.04


In [7]:
# Predictions for training and test sets
y_train_pred = Bayes_pipe.predict(X_train)
y_test_pred = Bayes_pipe.predict(X_valid)
y_test_prob = Bayes_pipe.predict_proba(X_valid)[:, 1]

# Compare the main classification metrics
metrics_comparison = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1 Score"],
    "Training": [
        accuracy_score(y_train, y_train_pred),
        precision_score(y_train, y_train_pred),
        recall_score(y_train, y_train_pred),
        f1_score(y_train, y_train_pred)
    ],
    "Validation": [
        accuracy_score(y_valid, y_test_pred),
        precision_score(y_valid, y_test_pred),
        recall_score(y_valid, y_test_pred),
        f1_score(y_valid, y_test_pred)
    ]
})

metrics_comparison.round(3)

,Metric,Training,Validation
0,Accuracy,0.846,0.816
1,Precision,0.501,0.434
2,Recall,0.659,0.636
3,F1 Score,0.570,0.516


### K-Nearest Neighbors

In [8]:
knn_pipeline = Pipeline([
    ("prep", preprocessor),
    ("model", KNeighborsClassifier(n_neighbors=5)),
])

fitted_knn = clone(knn_pipeline).fit(X_train, y_train)
knn_model = fitted_knn.named_steps["model"]
print(knn_model.weights)

session = X_train.iloc[[0]]
transformed_session = fitted_knn.named_steps["prep"].transform(session)
distances, indices = knn_model.kneighbors(transformed_session)

display(pd.DataFrame({
    "Distance": distances[0],
    "Purchased": y_train.iloc[indices[0]].to_numpy(),
}))

uniform


,Distance,Purchased
0,0.000000,False
1,1.620236,False
2,1.949654,False
3,2.000175,False
4,2.143301,False


## Support Vector Machine

In [9]:
svmlinear_pipe = Pipeline([
    ("prep", preprocessor),
    ("model", SVC(kernel="linear", C=1.0, class_weight="balanced"))
])

svmrbf_pipe = Pipeline([
    ("prep", preprocessor),
    ("model", SVC(
        kernel="rbf",
        C=1.0,
        gamma="scale",
        class_weight="balanced"
    ))
])

fitted_svmlinear = clone(svmlinear_pipe).fit(X_train, y_train)
fitted_svmrbf = clone(svmrbf_pipe).fit(X_train, y_train)

svmlinear_model = fitted_svmlinear.named_steps["model"]
svmrbf_model = fitted_svmrbf.named_steps["model"]
svm_support = pd.DataFrame({
    "Model": ["Linear SVM", "RBF SVM"],
    "Non-Buyer Support Vectors": [
        svmlinear_model.n_support_[0],
        svmrbf_model.n_support_[0]
    ],
    "Buyer Support Vectors": [
        svmlinear_model.n_support_[1],
        svmrbf_model.n_support_[1]
    ],
    "Total Support Vectors": [
        svmlinear_model.n_support_.sum(),
        svmrbf_model.n_support_.sum()
    ]
})
display(svm_support)


,Model,Non-Buyer Support Vectors,Buyer Support Vectors,Total Support Vectors
0,Linear SVM,2631,480,3111
1,RBF SVM,2585,564,3149


## Decision Tree

In [10]:
dt_pipe = Pipeline([
    ("prep", preprocessor),
    ("model", DecisionTreeClassifier(
        random_state=42,
        max_depth=5,
        min_samples_leaf=11,
    ))
])
fitted_dt = clone(dt_pipe).fit(X_train, y_train)

dt_model = fitted_dt.named_steps["model"]
dt_importance = pd.DataFrame({
    "Feature": fitted_dt.named_steps["prep"].get_feature_names_out(),
    "Importance": dt_model.feature_importances_
})

display(
    dt_importance.sort_values(
        "Importance",
        ascending=False
    ).head(10).round(3)
)


,Feature,Importance
8,num__PageValues,0.782
6,num__BounceRates,0.068
0,num__Administrative,0.035
17,cat__Month_Nov,0.035
5,num__ProductRelated_Duration,0.026
7,num__ExitRates,0.015
1,num__Administrative_Duration,0.010
4,num__ProductRelated,0.009
15,cat__Month_Mar,0.007
43,cat__Region_3,0.003


## Random Forest

In [11]:
rf_pipe = Pipeline([
    ("prep", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=400,
        random_state=42,
    ))
])

fitted_rf = clone(rf_pipe).fit(X_train, y_train)
rf_model = fitted_rf.named_steps["model"]

feature_names = fitted_rf.named_steps["prep"].get_feature_names_out()

importances = fitted_rf.named_steps["model"].feature_importances_

rf_importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
})

display(
    rf_importance.sort_values(
        "Importance",
        ascending=False
    ).head(10).round(3)
)


,Feature,Importance
8,num__PageValues,0.333
7,num__ExitRates,0.083
5,num__ProductRelated_Duration,0.078
4,num__ProductRelated,0.067
1,num__Administrative_Duration,0.053
6,num__BounceRates,0.049
0,num__Administrative,0.041
3,num__Informational_Duration,0.024
17,cat__Month_Nov,0.020
2,num__Informational,0.017


## Gradient Boosting

In [12]:
gb_pipe = Pipeline([
    ("prep", preprocessor),
    ("model", GradientBoostingClassifier(
        n_estimators=170,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    ))
])

fitted_gb = clone(gb_pipe).fit(X_train, y_train)
gb_model = fitted_gb.named_steps["model"]
gb_importance = pd.DataFrame({
    "Feature": fitted_gb.named_steps["prep"].get_feature_names_out(),
    "Importance": gb_model.feature_importances_
})

display(
    gb_importance.sort_values(
        "Importance",
        ascending=False
    ).head(10).round(3)
)


,Feature,Importance
8,num__PageValues,0.780
17,cat__Month_Nov,0.035
7,num__ExitRates,0.032
6,num__BounceRates,0.030
0,num__Administrative,0.026
5,num__ProductRelated_Duration,0.023
4,num__ProductRelated,0.019
1,num__Administrative_Duration,0.010
70,cat__VisitorType_Returning_Visitor,0.007
16,cat__Month_May,0.007


## Model Comparison

In [ ]:
model_results = []

pipelines = {
    "Logistic Regression": logistic_pipeline,
    "Naïve Bayes": Bayes_pipe,
    "K-Nearest Neighbors": knn_pipeline,
    "Linear SVM": svmlinear_pipe,
    "RBF SVM": svmrbf_pipe,
    "Decision Tree": dt_pipe,
    "Random Forest": rf_pipe,
    "Gradient Boosting": gb_pipe
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
f1_purchase = make_scorer(f1_score, pos_label=True, zero_division=0)

for model, pipeline in pipelines.items():
    scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring=f1_purchase,
        return_train_score=True,
    )

    model_results.append({
        "Model": model,
        "Mean training F1": scores["train_score"].mean(),
        "Mean CV F1": scores["test_score"].mean(),
    })
    
display(pd.DataFrame(model_results).round(3))